In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [2]:
print("su sakin ol")

su sakin ol


In [3]:
from fastapi import FastAPI
from pyngrok import ngrok
import nest_asyncio
import uvicorn

In [4]:
nest_asyncio.apply()  # Colab için gerekli

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Merhaba Sude, backend çalışıyor!"}

In [5]:
# Ucuz ve sorunsuz çözüm: thread ile başlatıyoruz
from threading import Thread

def start():
    uvicorn.run(app, host="0.0.0.0", port=8000)

Thread(target=start).start()

In [6]:
import requests
response = requests.get("http://127.0.0.1:8000/")
response.json()

INFO:     Started server process [30980]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:40288 - "GET / HTTP/1.1" 200 OK


{'message': 'Merhaba Sude, backend çalışıyor!'}

In [7]:
from fastapi import FastAPI, HTTPException
from threading import Thread
import nest_asyncio
import uvicorn

nest_asyncio.apply()  # Colab uyumluluğu

app = FastAPI()

# Basit "veritabanı"
users = []

# Ana sayfa
@app.get("/")
def home():
    return {"message": "Merhaba Sude, backend çalışıyor!"}

# Kullanıcı ekleme (Create)
@app.post("/users")
def create_user(name: str, age: int):
    user = {"id": len(users)+1, "name": name, "age": age}
    users.append(user)
    return {"message": "Kullanıcı eklendi", "user": user}

# Kullanıcı listeleme (Read)
@app.get("/users")
def list_users():
    return {"users": users}

# Kullanıcı silme (Delete)
@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    for user in users:
        if user["id"] == user_id:
            users.remove(user)
            return {"message": f"Kullanıcı {user_id} silindi."}
    raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")

# Server başlatma
def start():
    uvicorn.run(app, host="0.0.0.0", port=8000)

Thread(target=start).start()

INFO:     Started server process [30980]


In [8]:
import requests

# Kullanıcı ekleme
r1 = requests.post("http://127.0.0.1:8000/users", params={"name": "Sude", "age": 22})
print(r1.json())

# Kullanıcıları listeleme
r2 = requests.get("http://127.0.0.1:8000/users")
print(r2.json())

# Kullanıcı silme
r3 = requests.delete("http://127.0.0.1:8000/users/1")
print(r3.json())

# Tekrar listeleme
r4 = requests.get("http://127.0.0.1:8000/users")
print(r4.json())

INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:40300 - "POST /users?name=Sude&age=22 HTTP/1.1" 404 Not Found
{'detail': 'Not Found'}
INFO:     127.0.0.1:40306 - "GET /users HTTP/1.1" 404 Not Found
{'detail': 'Not Found'}
INFO:     127.0.0.1:40312 - "DELETE /users/1 HTTP/1.1" 404 Not Found
{'detail': 'Not Found'}
INFO:     127.0.0.1:40326 - "GET /users HTTP/1.1" 404 Not Found
{'detail': 'Not Found'}


In [9]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

users = []

@app.post("/users")
def create_user(name: str, age: int):
    user = {"id": len(users)+1, "name": name, "age": age}
    users.append(user)
    return {"message": "Kullanıcı eklendi", "user": user}

@app.get("/users")
def list_users():
    return {"users": users}

@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    for user in users:
        if user["id"] == user_id:
            users.remove(user)
            return {"message": f"Kullanıcı {user_id} silindi."}
    raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")

# TestClient ile Colab’da direkt test
client = TestClient(app)

# 1️⃣ Kullanıcı ekleme
r1 = client.post("/users", params={"name": "Sude", "age": 22})
print(r1.json())

# 2️⃣ Listeleme
r2 = client.get("/users")
print(r2.json())

# 3️⃣ Silme
r3 = client.delete("/users/1")
print(r3.json())

# 4️⃣ Tekrar listeleme
r4 = client.get("/users")
print(r4.json())

{'message': 'Kullanıcı eklendi', 'user': {'id': 1, 'name': 'Sude', 'age': 22}}
{'users': [{'id': 1, 'name': 'Sude', 'age': 22}]}
{'message': 'Kullanıcı 1 silindi.'}
{'users': []}


In [10]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

# FastAPI app
app = FastAPI()

# Basit “veritabanı” (liste)
users = []

# 1️⃣ Kullanıcı ekleme (Create)
@app.post("/users")
def create_user(name: str, age: int):
    user = {"id": len(users)+1, "name": name, "age": age}
    users.append(user)
    return {"message": "Kullanıcı eklendi", "user": user}

# 2️⃣ Kullanıcıları listeleme (Read)
@app.get("/users")
def list_users():
    return {"users": users}

# 3️⃣ Kullanıcı silme (Delete)
@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    for user in users:
        if user["id"] == user_id:
            users.remove(user)
            return {"message": f"Kullanıcı {user_id} silindi."}
    raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")

# 4️⃣ Kullanıcı güncelleme (Update)
@app.put("/users/{user_id}")
def update_user(user_id: int, name: str = None, age: int = None):
    for user in users:
        if user["id"] == user_id:
            if name is not None:
                user["name"] = name
            if age is not None:
                user["age"] = age
            return {"message": f"Kullanıcı {user_id} güncellendi", "user": user}
    raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")

# TestClient ile Colab’da çalıştırmak için
client = TestClient(app)

# 📌 Testler
# 1️⃣ Kullanıcı ekleme
r1 = client.post("/users", params={"name": "Sude", "age": 22})
print("Ekleme:", r1.json())

# 2️⃣ Listeleme
r2 = client.get("/users")
print("Listeleme:", r2.json())

# 3️⃣ Güncelleme
r3 = client.put("/users/1", params={"name": "Sude K.", "age": 23})
print("Güncelleme:", r3.json())

# 4️⃣ Silme
r4 = client.delete("/users/1")
print("Silme:", r4.json())

# 5️⃣ Tekrar listeleme
r5 = client.get("/users")
print("Son Listeleme:", r5.json())

Ekleme: {'message': 'Kullanıcı eklendi', 'user': {'id': 1, 'name': 'Sude', 'age': 22}}
Listeleme: {'users': [{'id': 1, 'name': 'Sude', 'age': 22}]}
Güncelleme: {'message': 'Kullanıcı 1 güncellendi', 'user': {'id': 1, 'name': 'Sude K.', 'age': 23}}
Silme: {'message': 'Kullanıcı 1 silindi.'}
Son Listeleme: {'users': []}


In [11]:
# 10 kullanıcı ekleyelim
for i in range(1, 11):
    client.post("/users", params={"name": f"Kullanıcı{i}", "age": 20+i})

In [12]:
list_users()

{'users': [{'id': 1, 'name': 'Kullanıcı1', 'age': 21},
  {'id': 2, 'name': 'Kullanıcı2', 'age': 22},
  {'id': 3, 'name': 'Kullanıcı3', 'age': 23},
  {'id': 4, 'name': 'Kullanıcı4', 'age': 24},
  {'id': 5, 'name': 'Kullanıcı5', 'age': 25},
  {'id': 6, 'name': 'Kullanıcı6', 'age': 26},
  {'id': 7, 'name': 'Kullanıcı7', 'age': 27},
  {'id': 8, 'name': 'Kullanıcı8', 'age': 28},
  {'id': 9, 'name': 'Kullanıcı9', 'age': 29},
  {'id': 10, 'name': 'Kullanıcı10', 'age': 30}]}

In [13]:
# 4. ve 5. kullanıcıyı güncelle
client.put("/users/4", params={"name": "Güncel 4", "age": 30})
client.put("/users/5", params={"name": "Güncel 5", "age": 31})

# 6. ve 7. kullanıcıyı sil
client.delete("/users/6")
client.delete("/users/7")

<Response [200 OK]>

In [14]:
list_users()

{'users': [{'id': 1, 'name': 'Kullanıcı1', 'age': 21},
  {'id': 2, 'name': 'Kullanıcı2', 'age': 22},
  {'id': 3, 'name': 'Kullanıcı3', 'age': 23},
  {'id': 4, 'name': 'Güncel 4', 'age': 30},
  {'id': 5, 'name': 'Güncel 5', 'age': 31},
  {'id': 8, 'name': 'Kullanıcı8', 'age': 28},
  {'id': 9, 'name': 'Kullanıcı9', 'age': 29},
  {'id': 10, 'name': 'Kullanıcı10', 'age': 30}]}

In [15]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
import sqlite3

app = FastAPI()

# SQLite veritabanı oluştur / bağlan
conn = sqlite3.connect(":memory:", check_same_thread=False)  # Colab için bellek içi DB
cursor = conn.cursor()

# Kullanıcı tablosu oluştur
cursor.execute("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER NOT NULL
)
""")
conn.commit()

# 1️⃣ Kullanıcı ekleme
@app.post("/users")
def create_user(name: str, age: int):
    cursor.execute("INSERT INTO users (name, age) VALUES (?, ?)", (name, age))
    conn.commit()
    user_id = cursor.lastrowid
    return {"message": "Kullanıcı eklendi", "user": {"id": user_id, "name": name, "age": age}}

# 2️⃣ Kullanıcıları listeleme
@app.get("/users")
def list_users():
    cursor.execute("SELECT id, name, age FROM users")
    users = [{"id": row[0], "name": row[1], "age": row[2]} for row in cursor.fetchall()]
    return {"users": users}

# 3️⃣ Kullanıcı silme
@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    cursor.execute("DELETE FROM users WHERE id=?", (user_id,))
    conn.commit()
    if cursor.rowcount == 0:
        raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")
    return {"message": f"Kullanıcı {user_id} silindi."}

# 4️⃣ Kullanıcı güncelleme
@app.put("/users/{user_id}")
def update_user(user_id: int, name: str = None, age: int = None):
    cursor.execute("SELECT id, name, age FROM users WHERE id=?", (user_id,))
    row = cursor.fetchone()
    if not row:
        raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")
    new_name = name if name else row[1]
    new_age = age if age else row[2]
    cursor.execute("UPDATE users SET name=?, age=? WHERE id=?", (new_name, new_age, user_id))
    conn.commit()
    return {"message": f"Kullanıcı {user_id} güncellendi", "user": {"id": user_id, "name": new_name, "age": new_age}}

# TestClient ile Colab’da test
client = TestClient(app)

# 📌 Testler
# Ekleme
client.post("/users", params={"name": "Sude", "age": 22})
client.post("/users", params={"name": "Ahmet", "age": 25})

# Listeleme
print("Listeleme:", client.get("/users").json())

# Güncelleme
client.put("/users/1", params={"name": "Sude K.", "age": 23})

# Silme
client.delete("/users/2")

# Son Listeleme
print("Son Listeleme:", client.get("/users").json())

Listeleme: {'users': [{'id': 1, 'name': 'Sude', 'age': 22}, {'id': 2, 'name': 'Ahmet', 'age': 25}]}
Son Listeleme: {'users': [{'id': 1, 'name': 'Sude K.', 'age': 23}]}


In [16]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
import sqlite3

app = FastAPI()

# Gerçek SQLite dosyası (Colab içinde de çalışır)
conn = sqlite3.connect("users.db", check_same_thread=False)
cursor = conn.cursor()

# Kullanıcı tablosu oluştur (ekstra alanlar ekledik)
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER NOT NULL,
    job TEXT,
    blood_group TEXT,
    religion TEXT,
    car_brand TEXT
)
""")
conn.commit()

# -------------------
# CRUD Endpointleri
# -------------------

# 1️⃣ Kullanıcı ekleme
@app.post("/users")
def create_user(name: str, age: int, job: str = "", blood_group: str = "", religion: str = "", car_brand: str = ""):
    cursor.execute(
        "INSERT INTO users (name, age, job, blood_group, religion, car_brand) VALUES (?, ?, ?, ?, ?, ?)",
        (name, age, job, blood_group, religion, car_brand)
    )
    conn.commit()
    user_id = cursor.lastrowid
    return {"message": "Kullanıcı eklendi", "user": {"id": user_id, "name": name, "age": age, "job": job, "blood_group": blood_group, "religion": religion, "car_brand": car_brand}}

# 2️⃣ Listeleme
@app.get("/users")
def list_users():
    cursor.execute("SELECT * FROM users")
    users = [{"id": row[0], "name": row[1], "age": row[2], "job": row[3], "blood_group": row[4], "religion": row[5], "car_brand": row[6]} for row in cursor.fetchall()]
    return {"users": users}

# 3️⃣ Silme
@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    cursor.execute("DELETE FROM users WHERE id=?", (user_id,))
    conn.commit()
    if cursor.rowcount == 0:
        raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")
    return {"message": f"Kullanıcı {user_id} silindi."}

# 4️⃣ Güncelleme
@app.put("/users/{user_id}")
def update_user(user_id: int, name: str = None, age: int = None, job: str = None, blood_group: str = None, religion: str = None, car_brand: str = None):
    cursor.execute("SELECT * FROM users WHERE id=?", (user_id,))
    row = cursor.fetchone()
    if not row:
        raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")

    # Eski değerleri al, yoksa güncel değerleri ata
    new_name = name if name else row[1]
    new_age = age if age else row[2]
    new_job = job if job else row[3]
    new_blood = blood_group if blood_group else row[4]
    new_religion = religion if religion else row[5]
    new_car = car_brand if car_brand else row[6]

    cursor.execute("UPDATE users SET name=?, age=?, job=?, blood_group=?, religion=?, car_brand=? WHERE id=?",
                   (new_name, new_age, new_job, new_blood, new_religion, new_car, user_id))
    conn.commit()
    return {"message": f"Kullanıcı {user_id} güncellendi", "user": {"id": user_id, "name": new_name, "age": new_age, "job": new_job, "blood_group": new_blood, "religion": new_religion, "car_brand": new_car}}

# -------------------
# TestClient ile hızlı test
# -------------------
client = TestClient(app)

# 10 farklı kullanıcı ekleyelim (manuel, farklı bilgilerle)
users_data = [
    {"name": "Sude", "age": 22, "job": "Mühendis", "blood_group": "A+", "religion": "None", "car_brand": "Tesla"},
    {"name": "Ahmet", "age": 25, "job": "Öğretmen", "blood_group": "B-", "religion": "Islam", "car_brand": "BMW"},
    {"name": "Elif", "age": 30, "job": "Doktor", "blood_group": "O+", "religion": "Hristiyan", "car_brand": "Mercedes"},
    {"name": "Mehmet", "age": 28, "job": "Pilot", "blood_group": "AB+", "religion": "Islam", "car_brand": "Audi"},
    {"name": "Ayşe", "age": 27, "job": "Avukat", "blood_group": "A-", "religion": "None", "car_brand": "Volvo"},
    {"name": "Can", "age": 35, "job": "Yazılımcı", "blood_group": "O-", "religion": "Yahudi", "car_brand": "Ford"},
    {"name": "Fatma", "age": 29, "job": "Hemşire", "blood_group": "B+", "religion": "Hindu", "car_brand": "Honda"},
    {"name": "Ali", "age": 32, "job": "Mühendis", "blood_group": "AB-", "religion": "Buddist", "car_brand": "Toyota"},
    {"name": "Deniz", "age": 26, "job": "Öğrenci", "blood_group": "A+", "religion": "None", "car_brand": "Hyundai"},
    {"name": "Veli", "age": 40, "job": "Şef", "blood_group": "B-", "religion": "Islam", "car_brand": "Peugeot"},
]

for u in users_data:
    client.post("/users", params=u)

# Listeleme
print("Tüm kullanıcılar:", client.get("/users").json())

Tüm kullanıcılar: {'users': [{'id': 1, 'name': 'Sude', 'age': 22, 'job': 'Mühendis', 'blood_group': 'A+', 'religion': 'None', 'car_brand': 'Tesla'}, {'id': 2, 'name': 'Ahmet', 'age': 25, 'job': 'Öğretmen', 'blood_group': 'B-', 'religion': 'Islam', 'car_brand': 'BMW'}, {'id': 3, 'name': 'Elif', 'age': 30, 'job': 'Doktor', 'blood_group': 'O+', 'religion': 'Hristiyan', 'car_brand': 'Mercedes'}, {'id': 4, 'name': 'Mehmet', 'age': 28, 'job': 'Pilot', 'blood_group': 'AB+', 'religion': 'Islam', 'car_brand': 'Audi'}, {'id': 5, 'name': 'Ayşe', 'age': 27, 'job': 'Avukat', 'blood_group': 'A-', 'religion': 'None', 'car_brand': 'Volvo'}, {'id': 6, 'name': 'Can', 'age': 35, 'job': 'Yazılımcı', 'blood_group': 'O-', 'religion': 'Yahudi', 'car_brand': 'Ford'}, {'id': 7, 'name': 'Fatma', 'age': 29, 'job': 'Hemşire', 'blood_group': 'B+', 'religion': 'Hindu', 'car_brand': 'Honda'}, {'id': 8, 'name': 'Ali', 'age': 32, 'job': 'Mühendis', 'blood_group': 'AB-', 'religion': 'Buddist', 'car_brand': 'Toyota'}, {

In [17]:
import json

users = client.get("/users").json()
print(json.dumps(users, indent=4, ensure_ascii=False))

{
    "users": [
        {
            "id": 1,
            "name": "Sude",
            "age": 22,
            "job": "Mühendis",
            "blood_group": "A+",
            "religion": "None",
            "car_brand": "Tesla"
        },
        {
            "id": 2,
            "name": "Ahmet",
            "age": 25,
            "job": "Öğretmen",
            "blood_group": "B-",
            "religion": "Islam",
            "car_brand": "BMW"
        },
        {
            "id": 3,
            "name": "Elif",
            "age": 30,
            "job": "Doktor",
            "blood_group": "O+",
            "religion": "Hristiyan",
            "car_brand": "Mercedes"
        },
        {
            "id": 4,
            "name": "Mehmet",
            "age": 28,
            "job": "Pilot",
            "blood_group": "AB+",
            "religion": "Islam",
            "car_brand": "Audi"
        },
        {
            "id": 5,
            "name": "Ayşe",
            "age": 27,
       

In [18]:
# Silme işlemleri
client.delete("/users/3")  # Elif
client.delete("/users/7")  # Fatma
client.delete("/users/9")  # Deniz

<Response [200 OK]>

In [19]:
list_users()

{'users': [{'id': 1,
   'name': 'Sude',
   'age': 22,
   'job': 'Mühendis',
   'blood_group': 'A+',
   'religion': 'None',
   'car_brand': 'Tesla'},
  {'id': 2,
   'name': 'Ahmet',
   'age': 25,
   'job': 'Öğretmen',
   'blood_group': 'B-',
   'religion': 'Islam',
   'car_brand': 'BMW'},
  {'id': 4,
   'name': 'Mehmet',
   'age': 28,
   'job': 'Pilot',
   'blood_group': 'AB+',
   'religion': 'Islam',
   'car_brand': 'Audi'},
  {'id': 5,
   'name': 'Ayşe',
   'age': 27,
   'job': 'Avukat',
   'blood_group': 'A-',
   'religion': 'None',
   'car_brand': 'Volvo'},
  {'id': 6,
   'name': 'Can',
   'age': 35,
   'job': 'Yazılımcı',
   'blood_group': 'O-',
   'religion': 'Yahudi',
   'car_brand': 'Ford'},
  {'id': 8,
   'name': 'Ali',
   'age': 32,
   'job': 'Mühendis',
   'blood_group': 'AB-',
   'religion': 'Buddist',
   'car_brand': 'Toyota'},
  {'id': 10,
   'name': 'Veli',
   'age': 40,
   'job': 'Şef',
   'blood_group': 'B-',
   'religion': 'Islam',
   'car_brand': 'Peugeot'}]}

In [20]:
# Güncelleme işlemleri
client.put("/users/2", params={"job": "Müdür", "car_brand": "Audi"})
client.put("/users/5", params={"blood_group": "O+"})
client.put("/users/8", params={"age": 33, "religion": "Islam"})

<Response [200 OK]>

In [21]:
list_users()

{'users': [{'id': 1,
   'name': 'Sude',
   'age': 22,
   'job': 'Mühendis',
   'blood_group': 'A+',
   'religion': 'None',
   'car_brand': 'Tesla'},
  {'id': 2,
   'name': 'Ahmet',
   'age': 25,
   'job': 'Müdür',
   'blood_group': 'B-',
   'religion': 'Islam',
   'car_brand': 'Audi'},
  {'id': 4,
   'name': 'Mehmet',
   'age': 28,
   'job': 'Pilot',
   'blood_group': 'AB+',
   'religion': 'Islam',
   'car_brand': 'Audi'},
  {'id': 5,
   'name': 'Ayşe',
   'age': 27,
   'job': 'Avukat',
   'blood_group': 'O+',
   'religion': 'None',
   'car_brand': 'Volvo'},
  {'id': 6,
   'name': 'Can',
   'age': 35,
   'job': 'Yazılımcı',
   'blood_group': 'O-',
   'religion': 'Yahudi',
   'car_brand': 'Ford'},
  {'id': 8,
   'name': 'Ali',
   'age': 33,
   'job': 'Mühendis',
   'blood_group': 'AB-',
   'religion': 'Islam',
   'car_brand': 'Toyota'},
  {'id': 10,
   'name': 'Veli',
   'age': 40,
   'job': 'Şef',
   'blood_group': 'B-',
   'religion': 'Islam',
   'car_brand': 'Peugeot'}]}

In [22]:
# Güncel kullanıcı listesi
import json

print("Güncel kullanıcılar:\n", json.dumps(client.get("/users").json(), indent=4))

Güncel kullanıcılar:
 {
    "users": [
        {
            "id": 1,
            "name": "Sude",
            "age": 22,
            "job": "M\u00fchendis",
            "blood_group": "A+",
            "religion": "None",
            "car_brand": "Tesla"
        },
        {
            "id": 2,
            "name": "Ahmet",
            "age": 25,
            "job": "M\u00fcd\u00fcr",
            "blood_group": "B-",
            "religion": "Islam",
            "car_brand": "Audi"
        },
        {
            "id": 4,
            "name": "Mehmet",
            "age": 28,
            "job": "Pilot",
            "blood_group": "AB+",
            "religion": "Islam",
            "car_brand": "Audi"
        },
        {
            "id": 5,
            "name": "Ay\u015fe",
            "age": 27,
            "job": "Avukat",
            "blood_group": "O+",
            "religion": "None",
            "car_brand": "Volvo"
        },
        {
            "id": 6,
            "name": "Can",

In [23]:
!pip install ipywidgets
from ipywidgets import widgets, VBox, HBox, Output
from IPython.display import display
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.0 MB/s eta 0:00:00


In [24]:
output = Output()
display(output)

Output()

In [25]:
# Kullanıcı ekleme widgetları
name_input = widgets.Text(description="İsim:")
age_input = widgets.IntText(description="Yaş:")
job_input = widgets.Text(description="Meslek:")
blood_input = widgets.Text(description="Kan grubu:")
religion_input = widgets.Text(description="Din:")
car_input = widgets.Text(description="Araba:")

add_button = widgets.Button(description="Kullanıcı Ekle")

# Listeleme butonu
list_button = widgets.Button(description="Kullanıcıları Listele")

# Güncelleme widgetları
update_id_input = widgets.IntText(description="ID:")
update_name_input = widgets.Text(description="Yeni İsim:")
update_age_input = widgets.IntText(description="Yeni Yaş:")
update_job_input = widgets.Text(description="Yeni Meslek:")
update_button = widgets.Button(description="Güncelle")

# Silme widgetları
delete_id_input = widgets.IntText(description="Silinecek ID:")
delete_button = widgets.Button(description="Kullanıcıyı Sil")

In [26]:
# Kullanıcı ekleme
def add_user(b):
    with output:
        response = client.post("/users", params={
            "name": name_input.value,
            "age": age_input.value,
            "job": job_input.value,
            "blood_group": blood_input.value,
            "religion": religion_input.value,
            "car_brand": car_input.value
        })
        print("Ekleme:", json.dumps(response.json(), indent=4))

add_button.on_click(add_user)

# Listeleme
def list_users_btn(b):
    with output:
        print("Kullanıcı Listesi:")
        print(json.dumps(client.get("/users").json(), indent=4))

list_button.on_click(list_users_btn)

# Güncelleme
def update_user_btn(b):
    with output:
        response = client.put(f"/users/{update_id_input.value}", params={
            "name": update_name_input.value or None,
            "age": update_age_input.value or None,
            "job": update_job_input.value or None
        })
        print("Güncelleme:", json.dumps(response.json(), indent=4))

update_button.on_click(update_user_btn)

# Silme
def delete_user_btn(b):
    with output:
        response = client.delete(f"/users/{delete_id_input.value}")
        print("Silme:", json.dumps(response.json(), indent=4))

delete_button.on_click(delete_user_btn)

In [27]:
display(
    VBox([
        widgets.Label("➤ Kullanıcı Ekleme"),
        name_input, age_input, job_input, blood_input, religion_input, car_input, add_button,
        widgets.Label("➤ Kullanıcıları Listele"),
        list_button,
        widgets.Label("➤ Kullanıcı Güncelleme"),
        update_id_input, update_name_input, update_age_input, update_job_input, update_button,
        widgets.Label("➤ Kullanıcı Silme"),
        delete_id_input, delete_button
    ])
)

In [29]:
# Gerekli kütüphaneler
!pip install fastapi uvicorn nest-asyncio pyngrok ipywidgets --quiet

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
import sqlite3
import nest_asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

# Colab için async loop
nest_asyncio.apply()

# -----------------
# Backend: FastAPI + SQLite
# -----------------
app = FastAPI()

# SQLite veritabanı (Colab'da dosya olarak kaydediyor)
conn = sqlite3.connect("users.db", check_same_thread=False)
cursor = conn.cursor()

# Kullanıcı tablosu oluştur (ekstra alanlar ekledik)
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER NOT NULL,
    job TEXT,
    blood_group TEXT,
    religion TEXT,
    car_brand TEXT
)
""")
conn.commit()

# CRUD Endpoints
@app.post("/users")
def create_user(name: str, age: int, job: str = "", blood_group: str = "", religion: str = "", car_brand: str = ""):
    cursor.execute(
        "INSERT INTO users (name, age, job, blood_group, religion, car_brand) VALUES (?, ?, ?, ?, ?, ?)",
        (name, age, job, blood_group, religion, car_brand)
    )
    conn.commit()
    user_id = cursor.lastrowid
    return {"message": "Kullanıcı eklendi", "user": {"id": user_id, "name": name, "age": age, "job": job, "blood_group": blood_group, "religion": religion, "car_brand": car_brand}}

@app.get("/users")
def list_users():
    cursor.execute("SELECT * FROM users")
    users = [{"id": row[0], "name": row[1], "age": row[2], "job": row[3], "blood_group": row[4], "religion": row[5], "car_brand": row[6]} for row in cursor.fetchall()]
    return {"users": users}

@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    cursor.execute("DELETE FROM users WHERE id=?", (user_id,))
    conn.commit()
    if cursor.rowcount == 0:
        raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")
    return {"message": f"Kullanıcı {user_id} silindi."}

@app.put("/users/{user_id}")
def update_user(user_id: int, name: str = None, age: int = None, job: str = None, blood_group: str = None, religion: str = None, car_brand: str = None):
    cursor.execute("SELECT * FROM users WHERE id=?", (user_id,))
    row = cursor.fetchone()
    if not row:
        raise HTTPException(status_code=404, detail="Kullanıcı bulunamadı")
    new_name = name if name else row[1]
    new_age = age if age else row[2]
    new_job = job if job else row[3]
    new_blood = blood_group if blood_group else row[4]
    new_religion = religion if religion else row[5]
    new_car = car_brand if car_brand else row[6]
    cursor.execute("UPDATE users SET name=?, age=?, job=?, blood_group=?, religion=?, car_brand=? WHERE id=?",
                   (new_name, new_age, new_job, new_blood, new_religion, new_car, user_id))
    conn.commit()
    return {"message": f"Kullanıcı {user_id} güncellendi", "user": {"id": user_id, "name": new_name, "age": new_age, "job": new_job, "blood_group": new_blood, "religion": new_religion, "car_brand": new_car}}

# -----------------
# TestClient ile Colab'da çalıştır
# -----------------
client = TestClient(app)

# -----------------
# Frontend: ipywidgets
# -----------------
# Giriş alanları
name_in = widgets.Text(description="İsim:")
age_in = widgets.IntText(description="Yaş:")
job_in = widgets.Text(description="Meslek:")
blood_in = widgets.Text(description="Kan Grubu:")
religion_in = widgets.Text(description="Din:")
car_in = widgets.Text(description="Araba:")

user_id_in = widgets.IntText(description="ID:")

# Çıktı alanı
output = widgets.Output()

# Butonlar
btn_add = widgets.Button(description="Kullanıcı Ekle", button_style='success')
btn_update = widgets.Button(description="Güncelle", button_style='warning')
btn_delete = widgets.Button(description="Sil", button_style='danger')
btn_list = widgets.Button(description="Listele", button_style='info')

def show_users():
    users = client.get("/users").json()
    with output:
        clear_output()
        print(json.dumps(users, indent=4))

def add_user(b):
    client.post("/users", params={"name": name_in.value, "age": age_in.value, "job": job_in.value,
                                  "blood_group": blood_in.value, "religion": religion_in.value, "car_brand": car_in.value})
    show_users()

def update_user_func(b):
    client.put(f"/users/{user_id_in.value}", params={"name": name_in.value or None, "age": age_in.value or None,
                                                    "job": job_in.value or None, "blood_group": blood_in.value or None,
                                                    "religion": religion_in.value or None, "car_brand": car_in.value or None})
    show_users()

def delete_user_func(b):
    client.delete(f"/users/{user_id_in.value}")
    show_users()

def list_user_func(b):
    show_users()

btn_add.on_click(add_user)
btn_update.on_click(update_user_func)
btn_delete.on_click(delete_user_func)
btn_list.on_click(list_user_func)

# Widgetları göster
display(widgets.VBox([widgets.HBox([name_in, age_in]), widgets.HBox([job_in, blood_in, religion_in, car_in]),
                      widgets.HBox([user_id_in, btn_add, btn_update, btn_delete, btn_list]), output]))

# Başlangıçta kullanıcıları göster
show_users()

In [30]:
# -----------------------------
# Filtreleme / Arama Frontend
# -----------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

# output alanı (filtreleme sonucu burada gösterilecek)
output_filter = widgets.Output()

# -----------------------------
# Giriş kutuları (hangi alanlarla filtreleme yapmak istediğimiz)
# -----------------------------
name_filter = widgets.Text(description="İsim:")
job_filter = widgets.Text(description="Meslek:")
blood_filter = widgets.Text(description="Kan Grubu:")
car_filter = widgets.Text(description="Araba:")

# -----------------------------
# Filtreleme butonu
# -----------------------------
btn_filter = widgets.Button(description="Filtrele", button_style='info')

# -----------------------------
# Filtreleme fonksiyonu
# -----------------------------
def filter_users(b):
    # 1️⃣ Tüm kullanıcıları al
    users = client.get("/users").json()["users"]

    # 2️⃣ Filtreleme işlemi: girilen kutular boş değilse filtre uygula
    filtered = []
    for user in users:
        if name_filter.value and name_filter.value.lower() not in user["name"].lower():
            continue
        if job_filter.value and job_filter.value.lower() not in user["job"].lower():
            continue
        if blood_filter.value and blood_filter.value.lower() not in user["blood_group"].lower():
            continue
        if car_filter.value and car_filter.value.lower() not in user["car_brand"].lower():
            continue
        filtered.append(user)

    # 3️⃣ Sonucu output alanına yazdır
    with output_filter:
        clear_output()
        if filtered:
            print(json.dumps({"users": filtered}, indent=4))
        else:
            print("Eşleşen kullanıcı bulunamadı.")

# -----------------------------
# Butona fonksiyon bağla
# -----------------------------
btn_filter.on_click(filter_users)

# -----------------------------
# Widgetları göster
# -----------------------------
display(widgets.VBox([
    widgets.HBox([name_filter, job_filter]),
    widgets.HBox([blood_filter, car_filter]),
    btn_filter,
    output_filter
]))

In [31]:
# -----------------------------
# Tam CRUD + Filtreleme Frontend
# -----------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

# output alanları
output_crud = widgets.Output()
output_filter = widgets.Output()

# -----------------------------
# CRUD Giriş kutuları
# -----------------------------
name_in = widgets.Text(description="İsim:")
age_in = widgets.IntText(description="Yaş:")
job_in = widgets.Text(description="Meslek:")
blood_in = widgets.Text(description="Kan Grubu:")
religion_in = widgets.Text(description="Din:")
car_in = widgets.Text(description="Araba:")
user_id_in = widgets.IntText(description="ID:")

# -----------------------------
# Filtreleme Giriş kutuları
# -----------------------------
name_filter = widgets.Text(description="İsim:")
job_filter = widgets.Text(description="Meslek:")
blood_filter = widgets.Text(description="Kan Grubu:")
car_filter = widgets.Text(description="Araba:")

# -----------------------------
# CRUD Butonları
# -----------------------------
btn_add = widgets.Button(description="Kullanıcı Ekle", button_style='success')
btn_update = widgets.Button(description="Güncelle", button_style='warning')
btn_delete = widgets.Button(description="Sil", button_style='danger')
btn_list = widgets.Button(description="Listele", button_style='info')

# Filtreleme butonu
btn_filter = widgets.Button(description="Filtrele", button_style='info')

# -----------------------------
# CRUD Fonksiyonları
# -----------------------------
def show_users():
    users = client.get("/users").json()
    with output_crud:
        clear_output()
        print(json.dumps(users, indent=4))

def add_user(b):
    client.post("/users", params={"name": name_in.value, "age": age_in.value, "job": job_in.value,
                                  "blood_group": blood_in.value, "religion": religion_in.value, "car_brand": car_in.value})
    show_users()

def update_user_func(b):
    client.put(f"/users/{user_id_in.value}", params={"name": name_in.value or None, "age": age_in.value or None,
                                                    "job": job_in.value or None, "blood_group": blood_in.value or None,
                                                    "religion": religion_in.value or None, "car_brand": car_in.value or None})
    show_users()

def delete_user_func(b):
    client.delete(f"/users/{user_id_in.value}")
    show_users()

def list_user_func(b):
    show_users()

# -----------------------------
# Filtreleme Fonksiyonu
# -----------------------------
def filter_users(b):
    users = client.get("/users").json()["users"]
    filtered = []
    for user in users:
        if name_filter.value and name_filter.value.lower() not in user["name"].lower():
            continue
        if job_filter.value and job_filter.value.lower() not in user["job"].lower():
            continue
        if blood_filter.value and blood_filter.value.lower() not in user["blood_group"].lower():
            continue
        if car_filter.value and car_filter.value.lower() not in user["car_brand"].lower():
            continue
        filtered.append(user)
    with output_filter:
        clear_output()
        if filtered:
            print(json.dumps({"users": filtered}, indent=4))
        else:
            print("Eşleşen kullanıcı bulunamadı.")

# -----------------------------
# Butonlara fonksiyonları bağla
# -----------------------------
btn_add.on_click(add_user)
btn_update.on_click(update_user_func)
btn_delete.on_click(delete_user_func)
btn_list.on_click(list_user_func)
btn_filter.on_click(filter_users)

# -----------------------------
# Widgetları göster
# -----------------------------
display(widgets.VBox([
    widgets.Label("🔹 CRUD İşlemleri"),
    widgets.HBox([name_in, age_in]),
    widgets.HBox([job_in, blood_in, religion_in, car_in]),
    widgets.HBox([user_id_in, btn_add, btn_update, btn_delete, btn_list]),
    output_crud,
    widgets.Label("🔹 Filtreleme"),
    widgets.HBox([name_filter, job_filter]),
    widgets.HBox([blood_filter, car_filter]),
    btn_filter,
    output_filter
]))

# Başlangıçta tüm kullanıcıları göster
show_users()